# P01. Perfil de reportes y lectura por bloques

3 de octubre de 2026, sesión 2

Preparado por el PhD Esteban Hernández, CyberColombia.

Estos ejercicios usan tres muestras reales de 1.000 filas. Las primeras filas sirven para probar código, sin representatividad estadística. Los archivos originales permanecen intactos. No se requiere GPU ni una cuenta externa. Instalar requirements.txt y requirements_pqrs.txt y seleccionar el kernel del mismo entorno.


**Entorno:** Ubuntu-26.04 sobre WSL. Clonar siempre `main` en `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. Activar `.venv` desde la raíz y abrir Jupyter desde `kit/`. Seguir [la guía WSL](../../docs/instalacion-wsl.md).

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "kit", Path.cwd() / "big-data-postgraduate" / "kit"]
KIT = next((p.resolve() for p in candidates if (p / "pqrs_talleres.py").exists()), None)
assert KIT is not None, "Abrir desde bigdata/, la raíz del repositorio, kit/ o kit/Notebooks/"
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=KIT / "salidas" / "notebooks", repeticiones=3)
a.out.mkdir(parents=True, exist_ok=True)


In [2]:
r = lab.perfil(a)
assert sum(x["filas"] for x in r) == 3000
assert all(x["columnas"] == 38 and x["sha256_coincide"] for x in r)
r

[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]


[{'archivo': 'Muestra_1000_2023_II.csv',
  'corte': '2023_II',
  'filas': 1000,
  'columnas': 38,
  'bytes': 752214,
  'memoria_dataframe_bloque_max_bytes': 1346842,
  'ancho_incorrecto': 0,
  'meses': {'7': 1000},
  'vacios': {'OBJECTID': 0,
   'periodo': 0,
   'mes': 0,
   'pqr_canal': 0,
   'pet_cod_depto': 2,
   'pet_cod_mpio': 2,
   'id_afec': 0,
   'afec_parentesco': 0,
   'afec_genero': 0,
   'afec_edad': 0,
   'afec_edadr': 0,
   'afec_educ': 0,
   'afec_regafiliacion': 0,
   'afec_getnico': 0,
   'afec_pobespecial': 0,
   'afec_cod_depto': 0,
   'afec_cod_mpio': 0,
   'ent_nombre': 0,
   'ent_tipovig_sns': 0,
   'ent_cod_sns': 0,
   'ent_alias_sn': 0,
   'ent_cod_depto': 0,
   'ent_cod_mpio': 0,
   'cod_macromot': 0,
   'macromotivo': 0,
   'cod_motgen': 0,
   'motivo_general': 0,
   'cod_motesp': 0,
   'motivo_especifico': 0,
   'cod_tipo_motesp': 56,
   'tipo_de_motivo_especifico': 56,
   'cod_subtipo_motesp': 442,
   'subtipo_de_motivo_especifico': 442,
   'patologia_1': 0,

## Comparación de memoria
Repetir con bloques de 100 y 1.000 filas. Comparar memoria del DataFrame, bytes en disco y RSS: representan medidas diferentes.

In [3]:
memoria = []
for size in [100, 500, 1000]:
    a.bloque = size
    perfil = lab.perfil(a)
    memoria.append({"bloque": size, "max_dataframe_bytes": max(x["memoria_dataframe_bloque_max_bytes"] for x in perfil)})
memoria

[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]
[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]
[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]


[{'bloque': 100, 'max_dataframe_bytes': 271361},
 {'bloque': 500, 'max_dataframe_bytes': 1346842},
 {'bloque': 1000, 'max_dataframe_bytes': 2693128}]

## Entrega
Diccionario de cinco campos, cobertura por corte, política para ausencias y presupuesto de recursos. Distinguir afectado, peticionario y reporte. `periodo` contiene el año y debe combinarse con `mes`. No usar `id_afec` como clave de reclamo.